# Progressão turno a turno — por AGENTE

Agrega os `turn_states.csv` e plota a **progressão média por rodada**, agrupando por **agente**
(greedy / long-term / random / …), não por cadeira P1/P2. Isso é o que torna a comparação justa
quando o torneio foi rodado com `randomize_seats=True` (cada agente joga metade em cada cadeira).

Cada linha de `turn_states.csv` é um snapshot `(jogo, turno, jogador)`. Reduzimos a **fim-de-rodada**
e tiramos a média entre os jogos, por agente.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TURN_STATES_PATH = None  # Opcional: aponte para um turn_states.csv específico.


def find_latest_turn_states_csv(root: Path) -> Path:
    base = root / 'artifacts' / 'tournaments'
    dirs = sorted(base.glob('*/'), key=lambda p: p.stat().st_mtime, reverse=True)
    for d in dirs:
        candidate = d / 'turn_states.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Nenhum turn_states.csv encontrado. Rode um torneio primeiro.')


turn_states_path = Path(TURN_STATES_PATH) if TURN_STATES_PATH else find_latest_turn_states_csv(ROOT)
print('Usando:', turn_states_path)

In [ ]:
df = pd.read_csv(turn_states_path)


def read_meta(csv_path):
    meta_path = csv_path.parent / 'metadata.json'
    if meta_path.exists():
        return json.loads(meta_path.read_text(encoding='utf-8')).get('tournament_config', {})
    return {}

meta = read_meta(turn_states_path)

# Garante a coluna 'agent' (novos artefatos já a têm; nos antigos, deriva do metadata).
if 'agent' not in df.columns:
    a1, a2 = meta.get('agent_p1', 'P1'), meta.get('agent_p2', 'P2')
    df['agent'] = df['player'].map({1: a1, 2: a2})

print('Agentes:', sorted(df['agent'].unique()),
      '| P1 =', meta.get('agent_p1'), '| P2 =', meta.get('agent_p2'),
      '| randomize_seats =', meta.get('randomize_seats'))
print('linhas:', len(df), '| jogos:', df['seed'].nunique(),
      '| rodadas:', int(df['round'].min()), '->', int(df['round'].max()))
df.head()

## Agregação: fim-de-rodada, média entre jogos, por agente

In [ ]:
# Fim-de-rodada: para cada (jogo, jogador, rodada) pega o snapshot de maior turno.
end_of_round = df.loc[df.groupby(['seed', 'player', 'round'])['turn'].idxmax()]

# Média entre jogos por (rodada, AGENTE).
agg = end_of_round.groupby(['round', 'agent']).mean(numeric_only=True).reset_index()
games_alive = end_of_round.groupby('round')['seed'].nunique()
agents = sorted(agg['agent'].unique())


def series(metric, agent):
    sub = agg[agg['agent'] == agent].set_index('round')[metric].sort_index()
    return sub.index, sub.values


def shared(metric):
    # Métrica compartilhada por jogo (ecossistema/suprimentos): média por rodada.
    sub = end_of_round.groupby('round')[metric].mean().sort_index()
    return sub.index, sub.values


agg.head()

## Recursos por agente (Sol e Plâncton)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for a in agents:
    x, y = series('sun', a); ax.plot(x, y, marker='o', label=f'{a} · Sol')
for a in agents:
    x, y = series('plankton', a); ax.plot(x, y, marker='x', linestyle='--', label=f'{a} · Plâncton')
ax.set_title('Recursos médios por rodada'); ax.set_xlabel('rodada'); ax.set_ylabel('recurso')
ax.legend(); plt.tight_layout(); plt.show()

## Score por agente

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for a in agents:
    x, y = series('score', a); ax.plot(x, y, marker='o', label=a)
ax.set_title('Score médio por rodada'); ax.set_xlabel('rodada'); ax.set_ylabel('score')
ax.legend(); plt.tight_layout(); plt.show()

## Usado vs produzido (Sol, acumulado)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for a in agents:
    x, y = series('spent_sun', a); ax.plot(x, y, marker='o', label=f'{a} · usado')
    x, y = series('produced_sun', a); ax.plot(x, y, linestyle='--', marker='.', label=f'{a} · produzido')
ax.set_title('Sol: usado vs produzido (acumulado)'); ax.set_xlabel('rodada'); ax.legend()
plt.tight_layout(); plt.show()

## Ocupação do board por camada, por agente

In [ ]:
fig, axes = plt.subplots(1, len(agents), figsize=(7 * len(agents), 5), squeeze=False)
for col, a in enumerate(agents):
    ax = axes[0][col]
    for z, label in [('corals_z0', 'fundo (z0)'), ('corals_z1', 'meio (z1)'), ('corals_z2', 'topo (z2)')]:
        x, y = series(z, a); ax.plot(x, y, marker='o', label=label)
    ax.set_title(f'Corais de {a} por camada'); ax.set_xlabel('rodada'); ax.set_ylabel('nº de corais'); ax.legend()
plt.tight_layout(); plt.show()

## Solos e mão de cartas (corais) por agente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for a in agents:
    x, y = series('soils_count', a); axes[0].plot(x, y, marker='o', label=a)
axes[0].set_title('Tiles de solo por agente'); axes[0].set_xlabel('rodada'); axes[0].set_ylabel('nº de solos'); axes[0].legend()
for a in agents:
    x, y = series('hand_size', a); axes[1].plot(x, y, marker='o', label=a)
axes[1].axhline(10, linestyle=':', color='k', label='teto (10)')
axes[1].set_title('Cartas de coral na mão'); axes[1].set_xlabel('rodada'); axes[1].set_ylabel('cartas'); axes[1].legend()
plt.tight_layout(); plt.show()

## Ecossistema e suprimentos (compartilhados por jogo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x, y = shared('temperature'); axes[0].plot(x, y, marker='o', color='tab:red')
axes[0].axhline(32.0, linestyle=':', color='k', label='crítico (32°C)')
axes[0].set_title('Temperatura média por rodada'); axes[0].set_xlabel('rodada'); axes[0].set_ylabel('°C'); axes[0].legend()

x, y = shared('soil_pile_remaining'); axes[1].plot(x, y, marker='o', color='tab:brown', label='pilha de solo')
x, y = shared('coral_deck_remaining'); axes[1].plot(x, y, marker='o', color='tab:green', label='baralho de corais')
axes[1].set_title('Suprimentos restantes'); axes[1].set_xlabel('rodada'); axes[1].set_ylabel('cartas/tiles'); axes[1].legend()
plt.tight_layout(); plt.show()

## Tabela: progressão média por rodada (primeiro agente)

In [ ]:
cols = ['sun', 'plankton', 'produced_sun', 'spent_sun', 'score', 'soils_count', 'hand_size',
        'corals_z0', 'corals_z1', 'corals_z2']
first = agents[0]
table = agg[agg['agent'] == first].set_index('round')[cols].round(2)
table.insert(0, 'jogos_vivos', games_alive)
print('Agente:', first)
table